# Notebook 3: Pythagoras Runs a Regression

**Act II — Power** | **Duration:** 55 min | **Register:** Semi-Formal

---

In Act I you discovered that regression *is* projection — a shadow cast by data onto the column space. Now that geometric lens gives you powers. In this notebook, three classical results that seemed arbitrary in your textbook will reveal themselves as trivial geometric facts.

**Core theorems:** SST = SSR + SSE, R² = cos²θ, Gauss-Markov, R² monotonicity.

**What you’ll need:**
- `regression_geometry.core` — the projection engine
- `regression_geometry.plots` — static visualizations
- `regression_geometry.data` — the Meridian dataset

*"Three squared lengths, one right angle — the rest is commentary."*

In [ ]:
# ============================================================
# Notebook 03: "Pythagoras Runs a Regression"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from regression_geometry.core import ColumnSpace, Projection, HatMatrix, demean
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

In [ ]:
# Load the Meridian dataset
mer = load_meridian()
y = mer["salary"].values
print(f"Meridian Analytics: {len(mer)} employees")
mer.head()

## Beat 1: The Pythagorean Theorem IS Variance Decomposition

Notebook 2 established the fundamental fact: the fitted values $\hat{y}$ and the residuals $e$ are **perpendicular**. Every regression produces a right angle.

Now look at what that right angle *gives* you.

### 🛑 PREDICT FIRST

Notebook 2 showed that $\hat{y}$ and $e$ are perpendicular. If you know the lengths $\|\hat{y}\|$ and $\|e\|$, can you determine $\|y\|$? **What classical theorem would you use?**

*Write your answer before running the next cell.*

In [ ]:
# Set up the salary ~ experience regression
X_exp = mer[["experience"]].values

cs_exp = ColumnSpace(X_exp, add_intercept=True)
proj_exp = cs_exp.project(y)

print("y = ŷ + e, and ŷ ⊥ e.")
print()
print(f"SST = {proj_exp.sst:.2f}")
print(f"SSR = {proj_exp.ssr:.2f}")
print(f"SSE = {proj_exp.sse:.2f}")
print(f"SSR + SSE = {proj_exp.ssr + proj_exp.sse:.2f}")
print()
print(f"Pythagorean check: SST ≈ SSR + SSE? {proj_exp.verify_pythagorean()}")

### Theorem Without Algebra #1

The decomposition $y = \hat{y} + e$ is a vector sum. Notebook 2 proved $\hat{y} \perp e$. The Pythagorean theorem immediately gives:

$$\|y\|^2 = \|\hat{y}\|^2 + \|e\|^2$$

After demeaning, these squared norms are *exactly* SST, SSR, and SSE:

$$\text{SST} = \text{SSR} + \text{SSE}$$

That’s it. The variance decomposition you memorized in your statistics course **is** the Pythagorean theorem applied to a right triangle in $\mathbb{R}^n$.

In [ ]:
# Visualize the right triangle with labeled sides and squared lengths
viz.plot_pythagorean_triangle(proj_exp)

In [ ]:
# Cross-check with statsmodels
model_exp = sm.OLS(y, sm.add_constant(X_exp)).fit()

print("Geometric computation vs. statsmodels:")
print(f"  SST: {proj_exp.sst:.2f}  vs  {model_exp.centered_tss:.2f}")
print(f"  SSR: {proj_exp.ssr:.2f}  vs  {model_exp.ess:.2f}")
print(f"  SSE: {proj_exp.sse:.2f}  vs  {model_exp.ssr:.2f}")
print()
print("(Note: statsmodels calls the residual sum of squares 'ssr' — confusing but correct.)")

## Beat 2: R² = cos²θ

Look at the right triangle again. The angle $\theta$ sits between $y$ (the hypotenuse) and $\hat{y}$ (the adjacent side). Trigonometry says:

$$\cos\theta = \frac{\|\hat{y}\|}{\|y\|} = \frac{\text{adjacent}}{\text{hypotenuse}}$$

Square both sides:

$$\cos^2\theta = \frac{\|\hat{y}\|^2}{\|y\|^2} = \frac{\text{SSR}}{\text{SST}} = R^2$$

**R² is literally the cosine squared of the angle between your data and your predictions.**

In [ ]:
# Visualize the angle and R²
viz.plot_r_squared_angle(proj_exp)

In [ ]:
# Verify R² = cos²θ
print(f"θ = {proj_exp.theta_degrees:.2f}°")
print(f"cos²θ = {np.cos(proj_exp.theta)**2:.6f}")
print(f"R²    = {proj_exp.r_squared:.6f}")
print(f"Match? {np.isclose(np.cos(proj_exp.theta)**2, proj_exp.r_squared)}")

In [ ]:
# The Meridian salary ~ experience regression
print(f"For salary ~ experience on Meridian data:")
print(f"  R² = {proj_exp.r_squared:.4f}")
print(f"  θ  = {proj_exp.theta_degrees:.1f}°")
print()
print(f"An R² of {proj_exp.r_squared:.2f} means the angle between your data and")
print(f"your predictions is about {proj_exp.theta_degrees:.0f}°.")
print(f"Does that feel like a good fit or a bad fit?")

## Beat 3: Gauss-Markov — "The Shortest Path"

The Gauss-Markov theorem says OLS is **BLUE** — the Best Linear Unbiased Estimator. Every textbook proves this with a half-page of matrix algebra. The geometry proves it in one sentence.

### Theorem Without Algebra #2

The orthogonal projection is the **closest point** in the subspace to $y$. Any other linear unbiased estimator $\tilde{\beta} = Cy$ produces a different point $\tilde{y} = X\tilde{\beta}$ in the column space. But $\hat{y}_{OLS}$ is the closest point to $y$ in that subspace — it has the smallest $\|e\|^2$, which means the smallest variance.

OLS doesn’t just find *a* projection — it finds **the** closest one. Any other linear unbiased estimator "reaches past" the closest point and grabs extra noise. That extra reach is wasted variance. This is the Gauss-Markov theorem, and the geometric proof is one picture.

In [ ]:
# Demonstrate: OLS gives the shortest residual
beta_ols = proj_exp.coefficients
e_ols = proj_exp.residuals

# Perturb the coefficients — still in the column space, but not the closest point
beta_other = beta_ols + np.array([1000, -50])
y_hat_other = sm.add_constant(X_exp) @ beta_other
e_other = y - y_hat_other

print(f"OLS residual norm:       ‖e_OLS‖ = {np.linalg.norm(e_ols):.2f}")
print(f"Other residual norm:     ‖e_other‖ = {np.linalg.norm(e_other):.2f}")
print()
print(f"OLS variance (‖e‖²/n):    {np.dot(e_ols, e_ols)/len(y):.2f}")
print(f"Other variance (‖e‖²/n):  {np.dot(e_other, e_other)/len(y):.2f}")
print()
print("The OLS residual is always shorter. That's the theorem.")

<details>
<summary><strong>Going Deeper: The algebraic Gauss-Markov proof</strong></summary>

For any other linear unbiased estimator $\tilde{\beta} = Cy$ with $CX = I$ (unbiasedness):

Write $C = (X^\top X)^{-1}X^\top + D$ where $DX = 0$.

Then:
$$\text{Var}(\tilde{\beta}) = \sigma^2 C C^\top = \sigma^2 [(X^\top X)^{-1} + DD^\top]$$

Since $DD^\top$ is positive semi-definite, $\text{Var}(\tilde{\beta}) \geq \text{Var}(\hat{\beta}_{OLS})$ in the matrix (Loewner) ordering.

Compare: one line of algebra versus one picture. The geometric argument — "orthogonal projection is closest" — does the same work without any matrix manipulation.

</details>

## Beat 4: R² Monotonicity

### Theorem Without Algebra #3

Adding a predictor **expands** the column space. A projection onto a bigger surface is at least as close to $y$ as a projection onto a smaller one. Therefore $R^2$ can only **increase** (or stay the same) when you add predictors.

Think of it this way: if the shadow falls on a line, it’s good. If the shadow falls on a plane *containing* that line, it can only be better — the plane offers more room for the shadow to land closer to the object.

In [ ]:
## Beat 5: The Workflow Trap — Polynomial Overfitting

You've just learned that adding predictors always increases $R^2$. Push this idea to its limit.

## Beat 5: The Workflow Trap — Polynomial Overfitting

You’ve just learned that adding predictors always increases $R^2$. Let’s push this idea to its limit.

In [ ]:
# Add polynomial terms of experience
exp = mer["experience"].values

print(f"{'Polynomial Degree':<20} {'R²':>8} {'θ':>8}")
print("-" * 38)
for degree in [1, 2, 3, 5, 8, 12]:
    X_poly = np.column_stack([exp**d for d in range(1, degree + 1)])
    cs_poly = ColumnSpace(X_poly, add_intercept=True)
    proj_poly = cs_poly.project(y)
    print(f"degree = {degree:<11} {proj_poly.r_squared:>8.4f} {proj_poly.theta_degrees:>7.1f}°")

print()
print("R² is climbing. θ is shrinking. Everything looks better...")

### 🛑 PREDICT FIRST

R² is climbing. The angle θ is shrinking. The projection is getting closer to $y$. **Is this model getting better? Will it make good predictions for a new employee?**

*Write your prediction before running the next cell.*

In [ ]:
# The reveal: split the data and check out-of-sample performance
np.random.seed(42)
n = len(mer)
train_mask = np.random.rand(n) < 0.7
test_mask = ~train_mask

y_train, y_test = y[train_mask], y[test_mask]
exp_train, exp_test = exp[train_mask], exp[test_mask]

print(f"{'Degree':<10} {'In-sample R²':>14} {'Out-of-sample R²':>18}")
print("-" * 44)
for degree in [1, 2, 3, 5, 8, 12]:
    X_train = np.column_stack([exp_train**d for d in range(1, degree + 1)])
    model_poly = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    r2_in = model_poly.rsquared

    X_test = np.column_stack([exp_test**d for d in range(1, degree + 1)])
    y_hat_test = sm.add_constant(X_test) @ model_poly.params
    ss_res = np.sum((y_test - y_hat_test)**2)
    ss_tot = np.sum((y_test - y_test.mean())**2)
    r2_out = 1 - ss_res / ss_tot

    print(f"{degree:<10} {r2_in:>14.4f} {r2_out:>18.4f}")

print()
print("In-sample R² keeps climbing. Out-of-sample R² collapses.")
print("The shadow got closer in THIS room — but it means nothing for the NEXT room.")

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| SST (total sum of squares) | `model.centered_tss` |
| SSR (explained sum of squares) | `model.ess` |
| SSE (residual sum of squares) | `model.ssr` ⚠️ confusing name |
| R² = cos²θ | `model.rsquared` |
| θ (angle between y and ŷ) | `np.arccos(np.sqrt(model.rsquared))` |

---

🔗 **Back to Statsmodels**

Here is the Rosetta Stone between geometric quantities and statsmodels names.

In [ ]:
# Full model: salary ~ experience + education + performance + gender
X_full = mer[["experience", "education", "performance", "gender"]].values
cs_full = ColumnSpace(X_full, add_intercept=True)
proj_full = cs_full.project(y)

model_full = sm.OLS(y, sm.add_constant(X_full)).fit()

print("Geometric name → statsmodels name")
print("=" * 55)
print(f"SST  → model.centered_tss  = {model_full.centered_tss:>15.2f}")
print(f"SSR  → model.ess           = {model_full.ess:>15.2f}")
print(f"SSE  → model.ssr           = {model_full.ssr:>15.2f}")
print(f"R²   → model.rsquared      = {model_full.rsquared:>15.6f}")
print(f"θ    → arccos(√R²)         = {np.degrees(np.arccos(np.sqrt(model_full.rsquared))):>14.2f}°")
print()
print("⚠️  statsmodels calls the RESIDUAL sum of squares 'ssr'.")
print("    And the REGRESSION (explained) sum of squares 'ess'.")
print("    This is backwards from most textbooks. Don't let it trip you up.")

### 🛑 DIAGNOSE FIRST

The full model (salary ~ experience + education + performance + gender) gives the R² printed above. Without looking at the next cell:

1. What is the angle θ between $y$ and $\hat{y}$?
2. Is SSR larger or smaller than SSE?
3. If you remove `gender` from the model, will θ increase or decrease?

*Write your answers, then verify below.*

In [ ]:
# Verify your answers
print(f"1. θ = {proj_full.theta_degrees:.2f}°")
print(f"2. SSR = {proj_full.ssr:.2f}, SSE = {proj_full.sse:.2f}")
if proj_full.ssr > proj_full.sse:
    print("   SSR > SSE")
else:
    print("   SSR < SSE")

# Without gender
X_no_gender = mer[["experience", "education", "performance"]].values
cs_no_gender = ColumnSpace(X_no_gender, add_intercept=True)
proj_no_gender = cs_no_gender.project(y)
print(f"3. θ without gender: {proj_no_gender.theta_degrees:.2f}° → with gender: {proj_full.theta_degrees:.2f}°")
print("   Removing a predictor can only increase θ (shrink the column space).")

✍️ **The Memo**

Write a memo (3 sentences) to Meridian’s CFO explaining why R² increased from the simple experience model to the degree-12 polynomial model, but you **don’t** recommend using the polynomial model.

**Forbidden words:** *overfitting, projection, column space, Pythagorean.*

*Your memo here:*

...

## Geometric Scoreboard

The Scoreboard now has **4 active gauges**: θ (angle), R² (coefficient of determination), tr(H)/n (average leverage), and ‖e‖/‖y‖ (relative residual norm).

In [ ]:
---
### Summary

**What you learned:**

- **Variance decomposition IS the Pythagorean theorem.** SST = SSR + SSE is $\|y\|^2 = \|\hat{y}\|^2 + \|e\|^2$ applied to a right triangle.
- **R² IS the cosine of an angle.** $R^2 = \cos^2\theta$ where θ is the angle between data and predictions.
- **OLS IS the closest point.** Gauss-Markov says orthogonal projection minimizes variance — because it minimizes distance.

**Key geometric insight:** ***R² = cos²θ — the proportion of variance explained is the square of the cosine of the angle between your data and your predictions.***

### Next

You've been treating the coefficient vector $\hat{\beta}$ as a single object. But each coefficient has its own story. What does it mean to "control for" a variable? Notebook 4 will peel the variables apart and show you exactly what each coefficient measures — using a theorem called Frisch-Waugh-Lovell that makes "controlling for" a geometric operation.

---

## Summary and Bridge

**What you learned:**

- **Variance decomposition IS the Pythagorean theorem.** SST = SSR + SSE is $\|y\|^2 = \|\hat{y}\|^2 + \|e\|^2$ applied to a right triangle.
- **R² IS the cosine of an angle.** $R^2 = \cos^2\theta$ where θ is the angle between data and predictions.
- **OLS IS the closest point.** Gauss-Markov says orthogonal projection minimizes variance — because it minimizes distance.
- **Closer isn’t always better.** R² monotonicity is a geometric fact, but overfitting is a geometric trap that *looks* like improvement.

---

**Next:** You’ve been treating the coefficient vector $\hat{\beta}$ as a single object. But each coefficient has its own story. What does it mean to "control for" a variable? Notebook 4 will peel the variables apart and show you exactly what each coefficient measures — using a theorem called Frisch-Waugh-Lovell that makes "controlling for" a geometric operation.